# Aprendizaje de Simbolo: `ParabolicMLPSymbol` con datos GRF (calor 2D)

Mismo experimento que `02_calor_1d.ipynb` en $\mathbb{T}^2$, con referencia $a(\mathbf{k})=|\mathbf{k}|^2$
(`HeatSymbol`, $n=2$, $m=2$, $\rho=n+1=3$ para el simbolo aprendido).

Reparametrizacion, $\kappa$ y funcion de perdida ($\mathcal{L}_{\text{traj}}$, $\mathcal{L}_{\text{par}}$,
$\mathcal{L}_{\text{class}}$): ver `02_calor_1d.ipynb`, introduccion y Setup.

# Setup
El codigo de cada seccion esta en `parabolic_symbol.learning.experiment.GRFExperiment`.

- **Datos:** GRF en $\mathbb{T}^2=[0,2\pi)^2$, malla de $128\times128$ puntos, $k_{\text{cap}} = 32$.
  Generacion y tiempos: ver `01_exploracion.ipynb`, seccion 2.1. Todos los pares $(k,t)$ con
  $|k|_\infty\le k_{\text{cap}}$, y $k_{\max}=k_{\text{cap}}$.
- **Particion, $\kappa$, $C_\alpha$, barrido, early stopping y seleccion:** como en `02_calor_1d.ipynb`
  (Setup y secciones 2 a 4), con $m=2$.
- `cuda` si esta disponible y `float64`.

In [1]:
from pathlib import Path

import torch

from parabolic_symbol.symbols.heat import HeatSymbol
from parabolic_symbol.symbols.fractional_laplacian import FractionalLaplacianSymbol
from parabolic_symbol.learning.experiment import GRFExperiment, summary_table
from parabolic_symbol.utils import default_device

device = default_device()
dtype = torch.float64
print(f"device = {device}  |  dtype = {dtype}")

RESULTS = Path("../results/03_calor_2d")   # tablas (CSV) y figuras (PNG)
RESULTS.mkdir(parents=True, exist_ok=True)

device = cuda  |  dtype = torch.float64


In [2]:
calor = GRFExperiment(
    reference=HeatSymbol(n=2), m=2.0, resolution=128, k_cap=32,
    results=RESULTS, dtype=dtype, device=device,
)

## 1. Datos de referencia

Ventana de $k$ resultante y numero de muestras por conjunto.

In [3]:
display(calor.load_data())

ENTRENAMIENTO
100 condiciones iniciales x 10 tiempos, resolucion 128x128: 21450 pares (k, t) con |k|_inf <= 32 -> 2,145,000 muestras
  ventana de k: 2145 k, |k|_inf <= 32, |k| in [0, 45.3]
PRUEBA


20 condiciones iniciales x 20 tiempos, resolucion 128x128: 42900 pares (k, t) con |k|_inf <= 32 -> 858,000 muestras
  ventana de k: 2145 k, |k|_inf <= 32, |k| in [0, 45.3]



k_cap = 32  ->  k_max = 32  (ventana de k: 2145 k, |k|_inf <= 32, |k| in [0, 45.3])


,muestras,k_unicos
conjunto,,
entrenamiento,1716000,1716
validacion,429000,429
prueba,858000,2145


## 2. $\kappa$ teorico y constantes $C_\alpha$

Banda $(c, C)$, $\kappa$ y tabla de $C_\alpha$: ver `02_calor_1d.ipynb`, seccion 2.

In [4]:
display(calor.estimate_constants())

estimate_symbol: 137,800 de 1,716,000 pares (k, t) con senal (tol=1e-10); 611 de 1716 k de entrenamiento
  ventana usada: 611 k, |k|_inf <= 21, |k| in [0, 21.4]


kappa_hat referencia = 2.0000
(c, C) de los datos  = (0.4500, 1.0976)   m = 2, ventana: 610 k, |k|_inf <= 21, |k| in [1, 21.4]
kappa teorico        = 3.1427


,C_alpha datos (x1.5),C_alpha oraculo (x1.5),k_usados
alpha,,,
"(0, 0)",1.496732,1.499268,611
"(0, 1)",3.320392,3.354102,467
"(1, 0)",3.320392,3.354102,484
"(0, 2)",3.000000,3.000000,357
"(1, 1)",1.496732,1.499268,295
"(2, 0)",3.000000,3.000000,386
"(0, 3)",1.496732,1.499268,269
"(1, 2)",1.496732,1.499268,178
"(2, 1)",1.496732,1.499268,188


## 3. Barrido de hiperparametros

Ejes del barrido: ver `02_calor_1d.ipynb`, seccion 3.

In [5]:
calor.sweep()

[  1/30] 2x32  drop[-] wd=0e+00 | rel_l2=0.4151 | 3812.1s


[  2/30] 2x32  drop[-] wd=1e-04 | rel_l2=0.4150 | 3811.8s


[  3/30] 2x64  drop[-] wd=0e+00 | rel_l2=0.4132 | 4447.5s


[  4/30] 2x64  drop[-] wd=1e-04 | rel_l2=0.2754 | 4448.6s


[W1015 09:32:17.298944410 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1757413376 bytes (free: 1289682944, total: 6141509632).


[  5/30] 2x128 drop[-] wd=0e+00 | rel_l2=0.0115 | 3053.2s


[  6/30] 2x128 drop[-] wd=1e-04 | rel_l2=0.0104 | 3117.3s


[  7/30] 3x32  drop[-] wd=0e+00 | rel_l2=0.0074 | 1347.9s


[  8/30] 3x32  drop[-] wd=1e-04 | rel_l2=0.0065 | 1149.1s


## 4. Resultados del barrido

In [ ]:
barrido, por_eje = calor.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

Criterio de seleccion: ver `02_calor_1d.ipynb`, seccion 4.

In [ ]:
display(calor.select())

### Metricas del modelo seleccionado

Reportes de trayectoria y de simbolo: ver `02_calor_1d.ipynb`, seccion 4. Aqui el simbolo se compara con $|\mathbf{k}|^2$.

In [ ]:
traj, sym = calor.metrics()
display(traj)
display(sym)

### Evaluacion en prueba

Errores en prueba: ver `02_calor_1d.ipynb`, seccion 4. El error en espacio fisico se mide en la malla de $128\times128$.

In [ ]:
display(calor.test())

## 5. Curvas de entrenamiento

Perdida y criterio de early stopping: ver `02_calor_1d.ipynb`, seccion 5.

In [ ]:
calor.curves()

## 6. Simbolo aprendido vs. referencia

Simbolo frente a $|\mathbf{k}|^2$ en $|k_i|\le k_{\max}$: curva contra $|k|$, paridad y error absoluto contra $|k|$ (el simbolo es radial), y un mapa del error relativo sobre $(k_1,k_2)$.

In [ ]:
calor.symbol_figure()

## 7. Certificacion de parabolicidad

Rejilla de certificacion: ver `02_calor_1d.ipynb`, seccion 7.

In [ ]:
display(calor.certification())

## 8. Clase del simbolo: $S^{m,\rho}$

Verificacion de la clase: ver `01_exploracion.ipynb`, seccion 5, y `02_calor_1d.ipynb`, seccion 8.

In [ ]:
display(calor.symbol_class())

## 9. Resultados (desde `modelo.pt`)

Carga `modelo.pt` (guardado por `select`), reconstruye el simbolo y regenera los datos; se puede ejecutar sola,
despues del Setup, sin volver a entrenar. Si no hay `modelo.pt`, la primera celda reentrena solo la configuracion
elegida (de `seleccion.csv`). Como leer cada figura: ver `02_calor_1d.ipynb`, seccion 9.

In [ ]:
from parabolic_symbol.learning import figures
from parabolic_symbol.learning.experiment import GRFExperiment, load_experiment_model

device = calor.device
if not (calor.results / "modelo.pt").exists():
    calor.retrain_selected()   # solo la configuracion elegida, sin barrido

In [ ]:
cargado = GRFExperiment.from_checkpoint(calor.results / "modelo.pt", device=device)   # configuracion guardada en modelo.pt
modelo = load_experiment_model(calor.results / "modelo.pt", device=device)
cargado.regenerate_data()   # entrenamiento, validacion y prueba: misma semilla, mismos datos

**Campo.** Primera condicion inicial de prueba: solucion exacta, la del simbolo aprendido y su diferencia.

In [ ]:
figures.field_figure(modelo, cargado.grf_test, cargado.results)

**Multiplicador.** $e^{-t a_\theta(k)}$, $e^{-t a(k)}$ y su diferencia sobre $(k,t)$ en la ventana de entrenamiento.

In [ ]:
figures.multiplier_figure(modelo, cargado.reference, cargado.results, K=cargado.k_max)

**Simbolo fuera de la ventana.** $a_\theta(k)$ frente a la referencia hasta $|k|=1000$, y $h_\theta(k)=a_\theta(k)/\langle k\rangle^m$ con la banda $(c,C)$.

In [ ]:
figures.symbol_extrapolation_figure(modelo, cargado.reference, cargado.results, k_window=cargado.k_max)

**Error contra $t$, con extrapolacion.** Error relativo $L^2$ en espacio fisico de las condiciones iniciales de prueba, hasta $t=3$.

In [ ]:
figures.extrapolation_error_figure(modelo, cargado.grf_test, cargado.results)

**Error del simbolo.** $|a_\theta(k)-a(k)|/|a(k)|$ en cada $k$ de entrenamiento y de validacion.

In [ ]:
figures.symbol_error_figure(modelo, cargado.reference, cargado.train_data, cargado.val_data, cargado.results)

**Espectro de potencia.** $\mathbb{E}|\hat u(t,k)|^2$ medio de las condiciones iniciales de prueba, por capa de $|k|$, en $t\in\{0.05,0.25,1\}$.

In [ ]:
figures.power_spectrum_figure(modelo, cargado.grf_test, cargado.results)